In [ ]:
# Install Dependencies
%pip install anthropic python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"


In [ ]:
def addUserMsg(messages, text):
    msg = {"role":"user", "content":text}
    messages.append(msg)

def addAssistantMsg(messages, text):
    msg = {"role":"assistant", "content":text}
    messages.append(msg)

def sendMsg(messages, text, systemPrompt=None, ignoreBegin=None, ignoreEnd=None):
    addUserMsg(messages,text)
    if ignoreBegin:
        addAssistantMsg(messages, ignoreBegin)
    params = {
        "model":model,
        "max_tokens":1000,
        "messages":messages,
    }
    if ignoreEnd:
        params["stop_sequences"] = [ignoreEnd]
    if systemPrompt:
        params["system"] = systemPrompt
    response = client.messages.create(**params)
    respText = response.content[0].text
    addAssistantMsg(messages,respText)
    return respText


In [ ]:
messages = []

sendMsg(messages, "What is quantum computing. Write in 1 sentence")


In [ ]:
sendMsg(messages, "Give another sentence")

In [ ]:
chat = []
while True:
    try:
        inp = input("Question?")
        print(">"+inp)
        r = sendMsg(chat, inp)
        print("----")
        print(r)
        print("----")
    except KeyboardInterrupt:
        break


In [ ]:
sendMsg([], "What is value of 4x+7=9 ?")

In [ ]:
systemPrompt = """
You are a teacher.
Initially give hints rather than complete solutions.
Patiently walk students through problems step by step.
Show solutions for similar problems as examples.
"""
sendMsg([], "What is value of 4x+7=9 ?", systemPrompt)

In [ ]:
sendMsg([], "Generate a 1 sentence Movie Idea")

In [ ]:
def sendStream(messages, text, callback):
    addUserMsg(messages, text)
    with client.messages.stream(
            model=model,
            max_tokens=1000,
            messages=messages
    ) as stream:
        for text in stream.text_stream:
            callback(text)
        return stream.get_final_message().content[0].text


In [ ]:
sendStream([], "Generate 1 sentence about planets", lambda t: print(t)) #, end=""))

In [ ]:
sendMsg([], "Rule to monitor EC2 instances as json code block")

In [ ]:
js = sendMsg([], "Rule to monitor EC2 instances as json code block", ignoreBegin="```json", ignoreEnd="```")
import json

json.loads(js.strip())